# SABR Option Pricing Demo

This notebook shows a minimal end-to-end example for the beta = 1 SABR project package.

- install or import the package
- run one plain Monte Carlo example
- run one conditional Monte Carlo example
- compare prices and standard errors with a simple plot


In [ ]:
# If you open this notebook in Google Colab, this cell installs the package from GitHub.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q git+https://github.com/PXYE-1029/SABR-Option-Pricing-Conditional-MC.git
else:
    print("If needed, install locally with: python3 -m pip install -e .")


In [ ]:
from pathlib import Path
import sys

# Allow imports when the notebook is opened from the local repository.
cwd = Path.cwd().resolve()
if (cwd / "src").exists() and str(cwd) not in sys.path:
    sys.path.insert(0, str(cwd))
elif (cwd.parent / "src").exists() and str(cwd.parent) not in sys.path:
    sys.path.insert(0, str(cwd.parent))

import matplotlib.pyplot as plt

from src.utils import EuropeanOption, SABRModelParameters
from src.mc_pricer import price_european_option_mc
from src.conditional_mc import price_european_option_conditional_mc


In [ ]:
parameters = SABRModelParameters(
    spot=100.0,
    initial_volatility=0.2,
    beta=1.0,
    vol_of_vol=0.4,
    correlation=-0.3,
    risk_free_rate=0.01,
)
option = EuropeanOption(strike=100.0, maturity=1.0, option_type="call")

mc_result = price_european_option_mc(
    parameters=parameters,
    option=option,
    n_steps=50,
    n_paths=5000,
    seed=123,
)
cmc_result = price_european_option_conditional_mc(
    parameters=parameters,
    option=option,
    n_steps=50,
    n_paths=5000,
    seed=123,
    integration_method="trapezoidal",
)

print(f"Plain MC price:        {mc_result.price:.6f}")
print(f"Plain MC standard err: {mc_result.standard_error:.6f}")
print(f"Conditional MC price:  {cmc_result.price:.6f}")
print(f"Conditional MC std err:{cmc_result.standard_error:.6f}")


In [ ]:
labels = ["Plain MC", "Conditional MC"]
prices = [mc_result.price, cmc_result.price]
standard_errors = [mc_result.standard_error, cmc_result.standard_error]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, prices, color=["tab:blue", "tab:orange"])
axes[0].set_title("Price Estimate")
axes[0].set_ylabel("Price")

axes[1].bar(labels, standard_errors, color=["tab:blue", "tab:orange"])
axes[1].set_title("Standard Error")
axes[1].set_ylabel("Standard error")

fig.suptitle("Plain MC vs Conditional MC")
plt.tight_layout()
plt.show()
